- run tests on GPU / CPU
- Maybe also f32 / f64

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '2'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import jax
# jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_x64", False)

In [ ]:
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import jax.numpy as jnp
import equinox as eqx

from rhmag.utils.pretest_evaluation import create_multilevel_df
from rhmag.model_interfaces.model_interface import ModelInterface

from rhmag.data_management import FINAL_MATERIALS, MaterialSet, DataSet
from rhmag.utils.data_plotting import plot_sequence_prediction, plot_hysteresis_prediction
from rhmag.utils.model_evaluation import reconstruct_model_from_file, plot_model_frequency_sweep, evaluate_cross_validation, get_exp_ids
from rhmag.utils.final_data_evaluation import (
    FINAL_MATERIALS,
    TestSet,
    ResultSet,
    predict_test_scenarios,
    validate_result_set,
    visualize_result_set,
    evaluate_test_scenarios,
    update_pareto_df,
    get_exp_ids_per_material,
    predict_test_scenarios_single_material
)
from rhmag.model_setup import setup_normalizer, setup_dataset

In [ ]:
model_type = "GRU8"

exp_ids = get_exp_ids(
    model_type=model_type,
    exp_name="final-reduced-features-f32",
    enforce_identical_exp_name=True,
)
exp_ids

In [ ]:
def generate_dummy_data(key, past_len, future_len, batch_size):
    key, subkey = jax.random.split(key, 2)
    subkeys = jax.random.split(subkey, 4)
    
    B_past = jax.random.normal(subkeys[0], shape=(batch_size, past_len))
    H_past = jax.random.normal(subkeys[2], shape=(batch_size, past_len))
    B_future = jax.random.normal(subkeys[1], shape=(batch_size, future_len))
    T = jax.random.normal(subkeys[3], shape=(batch_size,))

    return B_past, H_past, B_future, T, key

In [ ]:
import timeit

In [ ]:
key = jax.random.key(2)

past_len = 100
future_len = 900

wall_times = []

gpus = jax.devices()
cpu_tag = "cpu"

device = gpus[-1]

with jax.default_device(device):
    model = reconstruct_model_from_file('D_GRU8_final-reduced-features-f32_3d0f8de4_seed12')
    compiled_model = eqx.filter_jit(model)
    
    for batch_size in jnp.logspace(1, 5.5, 10, dtype=jnp.int32):
        
        B_past, H_past, B_future, T, key = generate_dummy_data(key, past_len, future_len, int(batch_size))
        
        t = timeit.Timer(lambda: compiled_model(B_past, H_past, B_future, T).block_until_ready())
        wall_time = t.repeat(repeat=20, number=1)
       
        wall_times.append(
            {
                "model_type": model_type,
                "batch_size": batch_size,
                "wall_time": wall_time,
                "past_len": past_len,
                "future_len": future_len,
                "device": device,
            }
        )

In [ ]:
wall_times

In [ ]:
pd.DataFrame(wall_times)